## 1. Install the llama stack client

In [40]:
%pip install llama_stack


[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 2. List available models

In [86]:
from llama_stack_client import LlamaStackClient

client = LlamaStackClient(base_url="http://ragathon-team-1-ragathon-team-1.apps.llama-rag-pool-b84hp.aws.rh-ods.com/")
client.models.list()

[Model(identifier='vllm-inference/mistral-small-24b-w8a8', metadata={}, api_model_type='llm', provider_id='vllm-inference', type='model', provider_resource_id='mistral-small-24b-w8a8', model_type='llm'),
 Model(identifier='granite-embedding-125m', metadata={'embedding_dimension': 768.0}, api_model_type='embedding', provider_id='sentence-transformers', type='model', provider_resource_id='ibm-granite/granite-embedding-125m-english', model_type='embedding'),
 Model(identifier='sentence-transformers/all-MiniLM-L6-v2', metadata={'embedding_dimension': 384.0}, api_model_type='embedding', provider_id='sentence-transformers', type='model', provider_resource_id='all-MiniLM-L6-v2', model_type='embedding')]

## 3. Register your Milvus vector database with LlamaStack

In [112]:
from llama_stack_client import Agent, AgentEventLogger, LlamaStackClient

client = LlamaStackClient(base_url="http://ragathon-team-1-ragathon-team-1.apps.llama-rag-pool-b84hp.aws.rh-ods.com/")

models = client.models.list()

# Select the first LLM and first embedding models
model_id = next(m for m in models if m.model_type == "llm").identifier
embedding_model_id = (
    em := next(m for m in models if m.model_type == "embedding")
).identifier
embedding_dimension = em.metadata["embedding_dimension"]

print(model_id)
print(embedding_model_id)

vllm-inference/llama-4-scout-17b-16e-w4a16
granite-embedding-125m


## 4. Import and run the KubeFlow Pipeline
Import the "[docling_convert_pipeline_compiled.yaml](./docling_convert_pipeline_compiled.yaml)" KubeFlow Pipeline into your pipeline server, then run the pipeline to insert your PDF documents into the vector database.

When running the pipeline, you can customize the following parameters:

- `base_url`: Base URL to fetch PDF files from
- `pdf_filenames`: Comma-separated list of PDF filenames to download and convert
- `num_workers`: Number of parallel workers
- `vector_db_id`: Milvus vector database ID
- `service_url`: Milvus service URL
- `embed_model_id`: Embedding model to use
- `max_tokens`: Maximum tokens per chunk
- `use_gpu`: Enable/disable GPU acceleration

Note: The compiled pipeline was generated by running `python docling_convert_pipeline.py`.

## 5. Prompt the LLM
Prompt the LLM with a question in relation to the documents inserted, and see it return accurate answers.

In [116]:
import uuid
client = LlamaStackClient(base_url="http://ragathon-team-1-ragathon-team-1.apps.llama-rag-pool-b84hp.aws.rh-ods.com")

vector_db_string_name = "red_bank_financial_kb_b"

vector_dbs = client.vector_dbs.list()
for vector_db in vector_dbs:
    if vector_db.vector_db_name == vector_db_string_name:
        vector_db_id = vector_db.identifier
        print("Using vector_db_id:", vector_db_id)

rag_agent = Agent(
    client,
    model=model_id,
    instructions="You are a helpful assistant working for Red Bank Financial, and provide answers to customer queries.",
    tools=[
        {
            "name": "builtin::rag/knowledge_search",
            "args": {"vector_db_ids": [vector_db_id]},
        }
    ],
)

prompt = "How can I change my address? I've moved to another address"
# prompt = "How can I call you?"
# prompt = "Tell me a bit about Red Bank Financial."
print("prompt>", prompt)

session_id = rag_agent.create_session(session_name=f"s{uuid.uuid4().hex}")

response = rag_agent.create_turn(
    messages=[{"role": "user", "content": prompt}],
    session_id=session_id,
    stream=True,
)

for log in AgentEventLogger().log(response):
    log.print()

Using vector_db_id: vs_1f1dd1b7-49ad-4ceb-8e8d-f0bf9afe2179
prompt> How can I change my address? I've moved to another address
inference> 
tool_execution> Tool:knowledge_search Args:{'query': 'change address Red Bank Financial'}
tool_execution> Tool:knowledge_search Response:[TextContentItem(text='knowledge_search tool found 5 chunks:\nBEGIN of knowledge_search tool results.\n', type='text'), TextContentItem(text="Result 1\nContent: 1. How can I update my personal details on the Red Bank Financial website?\n- Log in to Online Banking with your username and password.\n- Go to Profile & Settings → Personal Information .\n- Select the details you'd like to update (address, phone number, email, etc.).\n- Confirm the changes with your OTP (One-Time Password) sent to your registered device.\n2. How do I reset my Online Banking password?\n- On the login page, click 'Forgot Password?' .\n- Enter your registered email/phone number.\n- Follow the instructions to reset your password securely.\n3.

### Congratulations! You've successfully inserted your PDF documents via a KubeFlow Pipeline, and queried your RAG application using Llama Stack! 🎉🥳